<a href="https://colab.research.google.com/github/salimaliraja/NoviceResearcher/blob/main/Salim_Aliraja_DemoDay_final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Comprehensive Qualitative Analysis Toolkit
Features included:
- TF-IDF similarity to user-defined theories/attributes
- Keyword extraction (CountVectorizer)
- Theme clustering (KMeans)
- BERTopic (if installed)
- LDA topic modelling (gensim)
- Sentiment analysis (VADER)
- Naive automated summaries
- Network graph of themes (networkx + matplotlib)
- Visual dashboard functions (matplotlib)
- Streamlit GUI to run analysis interactively
- FastAPI endpoints to run analysis programmatically

Notes:
- Some libraries are optional (bertopic, gensim, pyLDAvis). The code checks and warns when missing.
- To run the Streamlit app: `streamlit run this_file.py --server.port=8501`
- To run the FastAPI app: `uvicorn this_file:app --reload --port 8000`

"""

import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import networkx as nx
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Optional packages
try:
    from bertopic import BERTopic
    BERTOPIC_AVAILABLE = True
except Exception:
    BERTOPIC_AVAILABLE = False

try:
    import gensim
    import gensim.corpora as corpora
    from gensim.models.ldamodel import LdaModel
    GENSIM_AVAILABLE = True
except Exception:
    GENSIM_AVAILABLE = False

try:
    import pyLDAvis
    import pyLDAvis.gensim_models
    PYLDAVIS_AVAILABLE = True
except Exception:
    PYLDAVIS_AVAILABLE = False

# Ensure NLTK resources
nltk.download('vader_lexicon')

# ----------------------------
# Data IO
# ----------------------------

def load_interview_data(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data)


def load_theories(filepath: str) -> Dict[str, List[str]]:
    with open(filepath, "r", encoding="utf-8") as f:
        theories = json.load(f)
    return theories

# ----------------------------
# TF-IDF similarity
# ----------------------------

def analyse_interviews(df: pd.DataFrame, theories: Dict[str, List[str]]):
    results = []
    vectorizer = TfidfVectorizer(stop_words="english")

    # Pre-fit vectorizer on all interview texts once for efficiency
    corpus_texts = df["text"].tolist()
    vectorizer.fit(corpus_texts + [" "])

    for theory_name, attributes in theories.items():
        for attribute in attributes:
            corpus = corpus_texts + [attribute]
            tfidf_matrix = vectorizer.transform(corpus)

            attr_vec = tfidf_matrix[-1]
            interview_vecs = tfidf_matrix[:-1]
            sims = cosine_similarity(attr_vec, interview_vecs)[0]

            for i, score in enumerate(sims):
                results.append({
                    "participant": df.loc[i, "participant"],
                    "theory": theory_name,
                    "attribute": attribute,
                    "similarity_score": float(score)
                })

    return pd.DataFrame(results)

# ----------------------------
# Keyword extraction
# ----------------------------

def extract_keywords(df: pd.DataFrame, top_n=20):
    cv = CountVectorizer(stop_words="english", ngram_range=(1,2))
    matrix = cv.fit_transform(df["text"])
    sums = np.array(matrix.sum(axis=0)).flatten()
    vocab = cv.get_feature_names_out()
    word_freq = list(zip(vocab, sums))
    keywords = sorted(word_freq, key=lambda x: x[1], reverse=True)[:top_n]
    return pd.DataFrame(keywords, columns=["keyword", "frequency"])

# ----------------------------
# Theme clustering (KMeans + PCA for plotting)
# ----------------------------

def cluster_themes(df: pd.DataFrame, k=4):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(df["text"])

    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X)
    df2 = df.copy()
    df2['cluster'] = labels

    # PCA projection for plotting
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X.toarray())
    df2['x'] = coords[:,0]
    df2['y'] = coords[:,1]

    return df2, model, vectorizer

# ----------------------------
# BERTopic
# ----------------------------

def topic_model_bertopic(texts: List[str], nr_topics=None):
    if not BERTOPIC_AVAILABLE:
        raise ImportError('BERTopic is not installed. Install with `pip install bertopic`')

    topic_model = BERTopic(nr_topics=nr_topics)
    topics, probs = topic_model.fit_transform(texts)
    return topic_model, topics, probs

# ----------------------------
# LDA (gensim)
# ----------------------------

def topic_model_lda(texts: List[str], num_topics=5):
    if not GENSIM_AVAILABLE:
        raise ImportError('Gensim is not installed. Install with `pip install gensim`')

    tokenized = [gensim.utils.simple_preprocess(t) for t in texts]
    id2word = corpora.Dictionary(tokenized)
    corpus = [id2word.doc2bow(text) for text in tokenized]

    lda = LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, random_state=42, passes=10)
    return lda, corpus, id2word

# ----------------------------
# Sentiment analysis (VADER)
# ----------------------------

def sentiment_scores(df: pd.DataFrame):
    sia = SentimentIntensityAnalyzer()
    sentiments = []

    for i, row in df.iterrows():
        score = sia.polarity_scores(row["text"])
        sentiments.append({
            "participant": row["participant"],
            "negative": score["neg"],
            "neutral": score["neu"],
            "positive": score["pos"],
            "compound": score["compound"]
        })

    return pd.DataFrame(sentiments)

# ----------------------------
# Naive summarisation
# ----------------------------

def summarise_text(df: pd.DataFrame, sentences=2):
    summaries = []

    for i, row in df.iterrows():
        # Very naive: split by sentence and take first N
        pieces = [s.strip() for s in row["text"].split('.') if s.strip()]
        summary = ('. '.join(pieces[:sentences]) + '.') if pieces else ''
        summaries.append({
            "participant": row["participant"],
            "summary": summary
        })

    return pd.DataFrame(summaries)

# ----------------------------
# Network graph of themes
# ----------------------------

def build_theme_network(df: pd.DataFrame, top_terms_per_cluster=5):
    # Use CountVectorizer to get term frequencies per cluster
    if 'cluster' not in df.columns:
        raise ValueError('Dataframe must have a "cluster" column. Run cluster_themes first.')

    clusters = df['cluster'].unique()
    cv = CountVectorizer(stop_words='english', ngram_range=(1,2))

    # Build bipartite graph: cluster <-> term
    G = nx.Graph()

    for c in clusters:
        texts = df[df['cluster'] == c]['text'].tolist()
        if not texts:
            continue
        X = cv.fit_transform(texts)
        sums = np.array(X.sum(axis=0)).flatten()
        vocab = cv.get_feature_names_out()
        term_freq = list(zip(vocab, sums))
        top_terms = sorted(term_freq, key=lambda x: x[1], reverse=True)[:top_terms_per_cluster]

        cluster_node = f'cluster_{c}'
        G.add_node(cluster_node, bipartite=0)
        for term, freq in top_terms:
            term_node = f'term:{term}'
            if not G.has_node(term_node):
                G.add_node(term_node, bipartite=1)
            G.add_edge(cluster_node, term_node, weight=int(freq))

    return G


def plot_theme_network(G, figsize=(10,8), with_labels=True):
    plt.figure(figsize=figsize)
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=with_labels, node_size=300, font_size=8)
    edges = nx.get_edge_attributes(G,'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edges)
    plt.show()

# ----------------------------
# Visual dashboards (matplotlib)
# ----------------------------

def plot_keyword_freq(keywords_df, top_n=20):
    df = keywords_df.head(top_n)
    plt.figure(figsize=(8,6))
    plt.barh(df['keyword'][::-1], df['frequency'][::-1])
    plt.xlabel('Frequency')
    plt.title('Top Keywords')
    plt.tight_layout()
    plt.show()


def plot_clusters(df):
    plt.figure(figsize=(8,6))
    for c in sorted(df['cluster'].unique()):
        subset = df[df['cluster']==c]
        plt.scatter(subset['x'], subset['y'], label=f'Cluster {c}', alpha=0.7)
    plt.legend()
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('Cluster projection (PCA)')
    plt.show()

# ----------------------------
# Full analysis pipeline
# ----------------------------

def run_full_analysis(interview_file, theory_file, k_clusters=4, lda_topics=5, use_bertopic=False):
    df = load_interview_data(interview_file)
    theories = load_theories(theory_file)

    similarity = analyse_interviews(df, theories)
    keywords = extract_keywords(df, top_n=50)
    clustered_df, kmodel, vectorizer = cluster_themes(df, k=k_clusters)
    sentiments = sentiment_scores(df)
    summaries = summarise_text(df)

    bertopic_model = None
    bertopic_info = None
    if use_bertopic:
        if BERTOPIC_AVAILABLE:
            bertopic_model, topics, probs = topic_model_bertopic(df['text'].tolist())
            bertopic_info = {'topics': topics, 'probs': probs}
        else:
            print('BERTopic not available; install `bertopic` if you want this feature')

    lda_model = None
    lda_vis = None
    if GENSIM_AVAILABLE:
        try:
            lda_model, corpus, id2word = topic_model_lda(df['text'].tolist(), num_topics=lda_topics)
            if PYLDAVIS_AVAILABLE:
                lda_vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word)
        except Exception as e:
            print('LDA modelling failed:', e)

    outputs = {
        'similarity_analysis': similarity,
        'keywords': keywords,
        'clustered_data': clustered_df,
        'sentiments': sentiments,
        'summaries': summaries,
        'bertopic': bertopic_info,
        'lda_model': lda_model,
        'lda_vis': lda_vis
    }

    return outputs


def save_outputs(outputs: dict, folder='outputs'):
    os.makedirs(folder, exist_ok=True)

    for name, value in outputs.items():
        if isinstance(value, pd.DataFrame):
            value.to_csv(os.path.join(folder, f'{name}.csv'), index=False)
        elif name == 'bertopic' and value is not None:
            # save topics as JSON
            with open(os.path.join(folder, 'bertopic_topics.json'), 'w', encoding='utf-8') as f:
                json.dump({'topics': str(value['topics'])}, f)
        elif name == 'lda_model' and value is not None:
            value.save(os.path.join(folder, 'lda_model.gensim'))
        elif name == 'lda_vis' and value is not None:
            pyLDAvis.save_html(value, os.path.join(folder, 'lda_vis.html'))

# ----------------------------
# Streamlit GUI
# ----------------------------
try:
    import streamlit as st
    STREAMLIT_AVAILABLE = True
except Exception:
    STREAMLIT_AVAILABLE = False

if STREAMLIT_AVAILABLE:
    def streamlit_app():
        st.title('Qualitative Analysis Toolkit')

        uploaded_interviews = st.file_uploader('Upload interviews.json', type=['json'])
        uploaded_theories = st.file_uploader('Upload theories.json', type=['json'])

        k_clusters = st.number_input('Number of clusters (KMeans)', min_value=2, max_value=12, value=4)
        use_bertopic = st.checkbox('Run BERTopic (optional)')
        lda_topics = st.number_input('Number of LDA topics', min_value=2, max_value=20, value=5)

        if st.button('Run Analysis'):
            if not uploaded_interviews or not uploaded_theories:
                st.error('Please upload both interviews and theories JSON files')
            else:
                interviews_path = os.path.join('temp_interviews.json')
                theories_path = os.path.join('temp_theories.json')
                with open(interviews_path, 'wb') as f:
                    f.write(uploaded_interviews.getbuffer())
                with open(theories_path, 'wb') as f:
                    f.write(uploaded_theories.getbuffer())

                outputs = run_full_analysis(interviews_path, theories_path, k_clusters, lda_topics, use_bertopic)
                save_outputs(outputs, folder='streamlit_outputs')

                st.success('Analysis complete. Downloads:')
                for name, val in outputs.items():
                    if isinstance(val, pd.DataFrame):
                        st.download_button(f'Download {name}.csv', val.to_csv(index=False).encode('utf-8'), file_name=f'{name}.csv')
                if outputs.get('lda_vis') is not None:
                    st.markdown('LDA visualization available in outputs folder (lda_vis.html)')

# ----------------------------
# FastAPI endpoints
# ----------------------------
try:
    from fastapi import FastAPI, UploadFile, File
    from fastapi.responses import FileResponse, JSONResponse
    FASTAPI_AVAILABLE = True
except Exception:
    FASTAPI_AVAILABLE = False

if FASTAPI_AVAILABLE:
    app = FastAPI(title='Qualitative Analysis API')

    @app.post('/run-analysis')
    async def run_analysis_api(interviews: UploadFile = File(...), theories: UploadFile = File(...), k_clusters: int = 4, lda_topics: int = 5, use_bertopic: bool = False):
        interviews_path = 'api_interviews.json'
        theories_path = 'api_theories.json'
        with open(interviews_path, 'wb') as f:
            f.write(await interviews.read())
        with open(theories_path, 'wb') as f:
            f.write(await theories.read())

        outputs = run_full_analysis(interviews_path, theories_path, k_clusters, lda_topics, use_bertopic)
        save_outputs(outputs, folder='api_outputs')

        # Return summary metadata
        meta = {k: (isinstance(v, pd.DataFrame) and v.shape or None) for k,v in outputs.items()}
        return JSONResponse({'status': 'completed', 'outputs': meta})

    @app.get('/download/{name}')
    def download_output(name: str):
        path = os.path.join('api_outputs', f'{name}.csv')
        if os.path.exists(path):
            return FileResponse(path)
        return JSONResponse({'error':'file not found'}, status_code=404)

# ----------------------------
# If running as script
# ----------------------------
if __name__ == '__main__':
    # Example usage -- adjust files as needed
    INTERVIEWS = 'interviews.json'
    THEORIES = 'theories.json'

    if os.path.exists(INTERVIEWS) and os.path.exists(THEORIES):
        outputs = run_full_analysis(INTERVIEWS, THEORIES, k_clusters=4, lda_topics=5, use_bertopic=False)
        save_outputs(outputs)
        print('Outputs saved to /outputs')
    else:
        print('Place interviews.json and theories.json in the same folder as this script and run again.')


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Place interviews.json and theories.json in the same folder as this script and run again.
